# xG Feature QA

This notebook is the maintained exploratory view over `features_xg`. It is meant for feature sanity checks, class balance inspection, and visually checking whether the table still reflects plausible handball shot geometry.

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import PercentFormatter

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'data' / 'hbl_raw.duckdb').exists():
            return candidate
    raise FileNotFoundError('Could not locate repo root containing data/hbl_raw.duckdb')

REPO_ROOT = find_repo_root()
DB_PATH = REPO_ROOT / 'data' / 'hbl_raw.duckdb'
con = duckdb.connect(str(DB_PATH), read_only=True)
DB_PATH

In [ ]:
df_xg = con.execute("""
    SELECT
        fixture_id,
        event_id,
        attack_type,
        sub_type,
        shooter_distance_to_goal,
        shooter_distance_to_goal_sq,
        shot_angle_to_goal,
        closest_defender_distance,
        goalkeeper_distance_to_goal,
        num_defenders_close,
        target
    FROM features_xg
""").fetchdf()

summary = pd.DataFrame({
    'rows': [len(df_xg)],
    'fixtures': [df_xg['fixture_id'].nunique()],
    'goal_rate': [df_xg['target'].mean()],
    'median_shooter_distance_m': [df_xg['shooter_distance_to_goal'].median()],
    'median_shot_angle_rad': [df_xg['shot_angle_to_goal'].median()],
})
summary.T.rename(columns={0: 'value'})

In [ ]:
distance_bins = pd.cut(
    df_xg['shooter_distance_to_goal'],
    bins=[-np.inf, 6, 9, 12, 15, np.inf],
    labels=['<6m', '6-9m', '9-12m', '12-15m', '15m+']
)
distance_profile = (
    df_xg.assign(distance_band=distance_bins.astype('string').fillna('UNKNOWN'))
    .groupby('distance_band', dropna=False)
    .agg(rows=('event_id', 'count'), goal_rate=('target', 'mean'))
    .reset_index()
)

fig, ax1 = plt.subplots(figsize=(11, 5))
ax2 = ax1.twinx()

ax1.bar(distance_profile['distance_band'], distance_profile['rows'], color='#4c78a8', alpha=0.85)
ax2.plot(distance_profile['distance_band'], distance_profile['goal_rate'], color='#f58518', marker='o', linewidth=2)
ax2.yaxis.set_major_formatter(PercentFormatter(1.0))

ax1.set_title('Shot volume and goal rate by shooter distance band')
ax1.set_xlabel('Shooter distance band')
ax1.set_ylabel('Rows')
ax2.set_ylabel('Goal rate')
plt.tight_layout()
plt.show()
distance_profile

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
for target, color, label in [(1, '#1f77b4', 'goal'), (0, '#d62728', 'no goal')]:
    values = df_xg.loc[df_xg['target'] == target, 'shooter_distance_to_goal'].dropna()
    ax.hist(values, bins=35, density=True, alpha=0.45, color=color, label=label)

ax.set_title('Shooter distance distribution by outcome')
ax.set_xlabel('Shooter distance to goal (m)')
ax.set_ylabel('Density')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
heatmap = (
    df_xg.groupby(['attack_type', 'sub_type'], dropna=False)
    .agg(goal_rate=('target', 'mean'), rows=('event_id', 'count'))
    .reset_index()
    .query('rows >= 30')
    .pivot(index='sub_type', columns='attack_type', values='goal_rate')
    .sort_index()
)

cmap = LinearSegmentedColormap.from_list('handball', ['#f4f4f4', '#74add1', '#d73027'])
fig, ax = plt.subplots(figsize=(12, 6))
image = ax.imshow(heatmap.fillna(np.nan), aspect='auto', cmap=cmap, vmin=0.3, vmax=0.9)

ax.set_xticks(range(len(heatmap.columns)))
ax.set_xticklabels(heatmap.columns, rotation=30, ha='right')
ax.set_yticks(range(len(heatmap.index)))
ax.set_yticklabels(heatmap.index)
ax.set_title('Goal rate by attack type and sub type (min 30 rows)')

for row_idx, row_name in enumerate(heatmap.index):
    for col_idx, col_name in enumerate(heatmap.columns):
        value = heatmap.loc[row_name, col_name]
        if pd.notna(value):
            ax.text(col_idx, row_idx, f'{value:.0%}', ha='center', va='center', fontsize=9, color='black')

fig.colorbar(image, ax=ax, shrink=0.85, label='Goal rate')
plt.tight_layout()
plt.show()
heatmap

In [ ]:
fixture_volume = (
    df_xg.groupby('fixture_id')
    .agg(rows=('event_id', 'count'), goal_rate=('target', 'mean'))
    .sort_values('rows', ascending=False)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(fixture_volume['rows'], bins=25, color='#54a24b', alpha=0.85)
ax.set_title('Distribution of xG rows per fixture')
ax.set_xlabel('Rows per fixture')
ax.set_ylabel('Fixture count')
plt.tight_layout()
plt.show()

fixture_volume.describe(include='all')